# Report-agent answer evaluation (gemma4 plans × contextual RAG × gpt-oss reference)

Full pipeline over a **fixed 3-case subset** of frozen `gemma4` query plans:

- `hms_borderline_01` (borderline)
- `hms_high_01`, `hms_high_02` (elevated / high)

1. **Load plans** from `query_agent_eval/outputs/plans/gemma4/`
2. **Contextual RAG** (Ollama-context hybrid, K=5) → freeze contexts
3. **Generate answers:** `gpt-oss:20b` = reference; local LLMs = candidates (same question + same contexts)
4. **RAGAS answer metrics** vs gpt-oss ground truth: faithfulness, answer_relevancy, answer_correctness, semantic_similarity

**Kernel:** **Python 3.12** (`python312`) — RAGAS lives here, not on 3.11/3.14.

**Runtime note:** with ~18 questions × 5 candidates, answer gen is manageable; RAGAS with local `gpt-oss:20b` is still ~15–20 min/row. All steps **resume from disk cache**. Edit `SELECTED_CASE_IDS` in the config cell to change the set.

## 0. Setup

In [2]:
import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

print("Python:", sys.version)
print("Executable:", sys.executable)

PKG = Path.cwd()
if PKG.name == "scripts":
    PKG = PKG.parent
if (PKG / "src").exists() and PKG.name == "report_agent_eval":
    ROOT = PKG
else:
    ROOT = Path(r"c:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval")

MAS = ROOT.parent
for p in (ROOT / "src", MAS / "knowledge_db_contextual_eval" / "src", MAS / "xai_rag_depression"):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import ragas
from report_agent_eval.answer import generate_answer, load_answers, now_iso, save_answers
from report_agent_eval.aggregate import run_aggregate
from report_agent_eval.config import cfg, path
from report_agent_eval.plans import load_question_records, records_by_case
from report_agent_eval.ragas_score import ANSWER_METRICS, backend_info, score_batch
from report_agent_eval.retrieval import (
    ContextualRetriever,
    context_texts,
    load_frozen_contexts,
    save_frozen_contexts,
)

print("Package root:", ROOT)
print("ragas:", ragas.__version__)
print("RAGAS backend:", backend_info())

Python: 3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]
Executable: c:\Users\MossabBatal\AppData\Local\Programs\Python\Python312\python.exe
Package root: c:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval
ragas: 0.2.12
RAGAS backend: backend=ollama model=gpt-oss:20b embed=nomic-embed-text


In [ ]:
# --------- run controls (edit here) ---------
# Exactly 3 cases: 1 borderline + 2 high
SELECTED_CASE_IDS = [
    "hms_borderline_01",
    "hms_high_01",
    "hms_high_02",
]
CANDIDATES = list(cfg("candidates", default=["gemma4", "llama3", "qwen2.5", "phi4", "mistral"]))
REFERENCE_MODEL = str(cfg("reference_model", default="gpt-oss:20b"))
PLAN_MODEL = str(cfg("plan_model", default="gemma4"))
FORCE_RETRIEVE = False
FORCE_ANSWERS = False
FORCE_RAGAS = False
RAGAS_BATCH_SIZE = int(cfg("ragas", "batch_size", default=2))  # small batches = frequent saves
# -------------------------------------------

OLLAMA = str(cfg("ollama", "base_url", default="http://localhost:11434")).rstrip("/")
r = requests.get(f"{OLLAMA}/api/tags", timeout=15)
r.raise_for_status()
installed = {m["name"] for m in r.json().get("models", [])}
need = [REFERENCE_MODEL, "nomic-embed-text"] + CANDIDATES
print("Ollama models check:")
for m in need:
    ok = any(n == m or n.startswith(m.split(":")[0] + ":") or n.startswith(m) for n in installed)
    print(f"  {'OK' if ok else 'MISSING':7} {m}")

print(f"\nPLAN={PLAN_MODEL}  REF={REFERENCE_MODEL}  CANDIDATES={CANDIDATES}")
print(f"SELECTED_CASE_IDS={SELECTED_CASE_IDS}")

Ollama models check:
  OK      gpt-oss:20b
  OK      nomic-embed-text
  OK      gemma4
  OK      llama3
  OK      qwen2.5
  OK      phi4
  OK      mistral

PLAN=gemma4  REF=gpt-oss:20b  CANDIDATES=['gemma4', 'llama3', 'qwen2.5', 'phi4', 'mistral']
SELECTED_CASE_IDS=['hms_low_01', 'hms_borderline_01', 'hms_high_01']


## 1. Load frozen gemma4 question plans

In [ ]:
records = load_question_records(PLAN_MODEL, limit=LIMIT_CASES)
by_case = records_by_case(records)
print(f"Questions: {len(records)}  |  Cases: {len(by_case)}")
print("By stratum:")
print(pd.Series([r.stratum for r in records]).value_counts())
print("\nSample:")
print(records[0])

Questions: 18  |  Cases: 3
Cases: ['hms_low_01', 'hms_borderline_01', 'hms_high_01']
By stratum:
low           6
borderline    6
high          6
Name: count, dtype: int64

Sample:
QuestionRecord(case_id='hms_low_01', stratum='low', qid='Q1', question='What is the appropriate non-diagnostic communication strategy for a student with a low screening risk (0.05 probability) to ensure they do not receive false reassurance?', retrieval_query='non-diagnostic communication strategies for low depression/anxiety screening risk', question_type='safety_communication')


## 2. Phase 1a — Contextual RAG (freeze contexts once per question)

Uses the `knowledge_db_contextual_eval` winner: **Ollama-context hybrid, K=5**. Contexts are shared by every answer model.

In [5]:
retriever = ContextualRetriever()
total = len(records)
reused = 0
fetched = 0
for i, rec in enumerate(records, start=1):
    if not FORCE_RETRIEVE and load_frozen_contexts(rec.case_id, rec.qid) is not None:
        reused += 1
        if i % 50 == 0 or i == total:
            print(f"[{i}/{total}] cached…", flush=True)
        continue
    hits = retriever.retrieve(rec.retrieval_query)
    save_frozen_contexts(
        rec.case_id,
        rec.qid,
        {
            "case_id": rec.case_id,
            "stratum": rec.stratum,
            "qid": rec.qid,
            "question": rec.question,
            "retrieval_query": rec.retrieval_query,
            "question_type": rec.question_type,
            "condition": retriever.condition,
            "mode": retriever.mode,
            "top_k": retriever.top_k,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "hits": hits,
        },
    )
    fetched += 1
    print(f"[{i}/{total}] {rec.case_id} {rec.qid} -> {len(hits)} chunks", flush=True)

print(f"Retrieval done. fetched={fetched} reused={reused} total={total}")

[18/18] cached…
Retrieval done. fetched=0 reused=18 total=18


## 3. Phase 1b — Generate answers (gpt-oss reference + candidate LLMs)

Identical prompt + identical frozen contexts for every model. `gpt-oss:20b` answers become RAGAS `ground_truth`.

In [6]:
models = [REFERENCE_MODEL] + [m for m in CANDIDATES if m != REFERENCE_MODEL]
print("Generation order:", models)

done = 0
skipped_ctx = 0
total_jobs = len(models) * len(records)

for model in models:
    for case_id, case_recs in by_case.items():
        store = {} if FORCE_ANSWERS else load_answers(model, case_id)
        store.setdefault("model", model)
        store.setdefault("case_id", case_id)
        answers = store.setdefault("answers", {})
        changed = False
        for rec in case_recs:
            done += 1
            if (
                not FORCE_ANSWERS
                and rec.qid in answers
                and (answers[rec.qid].get("answer") or "").strip()
            ):
                continue
            frozen = load_frozen_contexts(rec.case_id, rec.qid)
            if frozen is None:
                skipped_ctx += 1
                print(f"[{done}/{total_jobs}] {model} {case_id} {rec.qid} SKIP (no context)", flush=True)
                continue
            contexts = context_texts(frozen)
            text, error = generate_answer(model, rec.question, contexts)
            answers[rec.qid] = {
                "qid": rec.qid,
                "question": rec.question,
                "question_type": rec.question_type,
                "stratum": rec.stratum,
                "answer": text,
                "error": error,
                "n_contexts": len(contexts),
                "timestamp_utc": now_iso(),
            }
            changed = True
            status = "ok" if text else f"FAIL({error or 'empty'})"
            print(f"[{done}/{total_jobs}] {model} {case_id} {rec.qid} -> {status}", flush=True)
        if changed or FORCE_ANSWERS:
            save_answers(model, case_id, store)

print(f"Answers done. jobs={done} skipped_missing_ctx={skipped_ctx}")

Generation order: ['gpt-oss:20b', 'gemma4', 'llama3', 'qwen2.5', 'phi4', 'mistral']
[1/108] gpt-oss:20b hms_low_01 Q1 -> ok
[2/108] gpt-oss:20b hms_low_01 Q2 -> ok
[3/108] gpt-oss:20b hms_low_01 Q3 -> ok
[4/108] gpt-oss:20b hms_low_01 Q4 -> ok
[5/108] gpt-oss:20b hms_low_01 Q5 -> ok
[6/108] gpt-oss:20b hms_low_01 Q6 -> ok
[19/108] gemma4 hms_low_01 Q1 -> ok
[20/108] gemma4 hms_low_01 Q2 -> ok
[21/108] gemma4 hms_low_01 Q3 -> ok
[22/108] gemma4 hms_low_01 Q4 -> ok
[23/108] gemma4 hms_low_01 Q5 -> ok
[24/108] gemma4 hms_low_01 Q6 -> ok
[25/108] gemma4 hms_borderline_01 Q1 -> ok
[26/108] gemma4 hms_borderline_01 Q2 -> ok
[27/108] gemma4 hms_borderline_01 Q3 -> ok
[28/108] gemma4 hms_borderline_01 Q4 -> ok
[29/108] gemma4 hms_borderline_01 Q5 -> ok
[30/108] gemma4 hms_borderline_01 Q6 -> ok
[31/108] gemma4 hms_high_01 Q1 -> ok
[32/108] gemma4 hms_high_01 Q2 -> ok
[33/108] gemma4 hms_high_01 Q3 -> ok
[34/108] gemma4 hms_high_01 Q4 -> ok
[35/108] gemma4 hms_high_01 Q5 -> ok
[36/108] gemma4 h

## 4. Phase 2 — RAGAS answer metrics (candidate vs gpt-oss reference)

Saves after every batch → safe to interrupt and re-run this cell.

In [7]:
def _row_id(model: str, case_id: str, qid: str) -> str:
    return f"{model}::{case_id}::{qid}"


ref_cache: dict[str, dict] = {}


def ref_answers(case_id: str) -> dict:
    if case_id not in ref_cache:
        ref_cache[case_id] = load_answers(REFERENCE_MODEL, case_id).get("answers", {})
    return ref_cache[case_id]


rows = []
for model in CANDIDATES:
    for rec in records:
        cand = load_answers(model, rec.case_id).get("answers", {}).get(rec.qid, {})
        answer = (cand.get("answer") or "").strip()
        gt = (ref_answers(rec.case_id).get(rec.qid, {}).get("answer") or "").strip()
        if not answer or not gt:
            continue
        frozen = load_frozen_contexts(rec.case_id, rec.qid)
        if frozen is None:
            continue
        contexts = context_texts(frozen) or ["No retrieved context available."]
        rows.append(
            {
                "row_id": _row_id(model, rec.case_id, rec.qid),
                "model": model,
                "case_id": rec.case_id,
                "qid": rec.qid,
                "stratum": rec.stratum,
                "question_type": rec.question_type,
                "question": rec.question,
                "answer": answer,
                "contexts": contexts,
                "ground_truth": gt,
            }
        )

dataset = pd.DataFrame(rows)
print(f"Scorable rows: {len(dataset)}  (need candidate + gpt-oss reference + context)")
if not dataset.empty:
    display(dataset.groupby("model").size().rename("n").to_frame())

Scorable rows: 90  (need candidate + gpt-oss reference + context)


,n
model,
gemma4,18
llama3,18
mistral,18
phi4,18
qwen2.5,18


In [8]:
out_csv = path("outputs_dir") / "ragas_per_question.csv"
path("outputs_dir").mkdir(parents=True, exist_ok=True)

existing = pd.DataFrame()
scored_ids: set[str] = set()
if out_csv.exists() and not FORCE_RAGAS:
    existing = pd.read_csv(out_csv)
    if "row_id" in existing.columns:
        scored_ids = set(existing["row_id"].astype(str))

if dataset.empty:
    print("Nothing to score — run Phase 1b first.")
else:
    pending = dataset[~dataset["row_id"].astype(str).isin(scored_ids)]
    print(
        f"RAGAS {backend_info()}\n"
        f"Already scored: {len(scored_ids)} | Pending: {len(pending)} | batch={RAGAS_BATCH_SIZE}",
        flush=True,
    )
    combined = existing.copy()
    n = len(pending)
    if n == 0:
        print("All pending rows already scored.")
    else:
        n_batches = (n + RAGAS_BATCH_SIZE - 1) // RAGAS_BATCH_SIZE
        for start in range(0, n, RAGAS_BATCH_SIZE):
            batch = pending.iloc[start : start + RAGAS_BATCH_SIZE]
            bi = start // RAGAS_BATCH_SIZE + 1
            print(f"  batch {bi}/{n_batches} ({len(batch)} rows)…", flush=True)
            scored = score_batch(batch)
            combined = (
                pd.concat([combined, scored], ignore_index=True) if not combined.empty else scored
            )
            combined.drop_duplicates(subset=["row_id"], keep="last", inplace=True)
            combined.to_csv(out_csv, index=False)
            ok = combined[ANSWER_METRICS].notna().any(axis=1).sum()
            print(f"    saved {len(combined)} rows ({ok} with metrics)", flush=True)
        print("RAGAS scoring pass finished:", out_csv)

RAGAS backend=ollama model=gpt-oss:20b embed=nomic-embed-text
Already scored: 2 | Pending: 88 | batch=8
  batch 1/11 (8 rows)…


c:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\src\report_agent_eval\ragas_score.py:56: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model=model, base_url=base_url, temperature=0)
c:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\src\report_agent_eval\ragas_score.py:57: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = Ollam

Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

    saved 10 rows (10 with metrics)
  batch 2/11 (8 rows)…


Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

...

    saved 74 rows (74 with metrics)
  batch 10/11 (8 rows)…


Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

    saved 82 rows (82 with metrics)
  batch 11/11 (8 rows)…


Evaluating:   0%|          | 0/32 [00:00<?, ?it/s]

    saved 90 rows (90 with metrics)
RAGAS scoring pass finished: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\ragas_per_question.csv


## 5. Aggregate — tables, charts, paper markdown

Safe to re-run anytime as more RAGAS rows accumulate.

In [9]:
csv_path = path("outputs_dir") / "ragas_per_question.csv"
if not csv_path.exists():
    print("No ragas_per_question.csv yet.")
else:
    scored = pd.read_csv(csv_path)
    print(f"Rows on disk: {len(scored)}")
    display(scored.groupby("model")[ANSWER_METRICS].mean().round(4))
    written = run_aggregate()
    for k, v in written.items():
        print(f"  {k}: {v}")
    paper = path("outputs_dir") / "report_answer_quality_for_paper.md"
    if paper.exists():
        print("\n--- paper md (head) ---\n")
        print("\n".join(paper.read_text(encoding="utf-8").splitlines()[:40]))

Rows on disk: 90


,faithfulness,answer_relevancy,answer_correctness,semantic_similarity
model,,,,
gemma4,0.8408,0.5597,0.5543,0.9295
llama3,0.4588,0.7037,0.5478,0.9369
mistral,0.6244,0.7204,0.5148,0.9338
phi4,0.5487,0.6808,0.6034,0.9435
qwen2.5,0.3665,0.7227,0.5391,0.9275


  by_model: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\ragas_by_model.csv
  long: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\per_question_long.csv
  by_stratum: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\ragas_by_model_stratum.csv
  ragas_answer_metrics_by_model: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\charts\ragas_answer_metrics_by_model.svg
  ragas_leaderboard: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\charts\ragas_leaderboard.svg
  paper_md: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\report_answer_quality_for_paper.md

--- paper md (head) ---

# Report Agent Answer Quality - RAGAS (scientific results)

Each evaluated local LLM answered the frozen `gemma4` query-plan questions from identical contextual-RAG evidence (Ollama-context hybrid, K=5). Answers are compared to `gpt-oss:20b` reference answers using RAGAS answer-on

--- paper md (head) ---

# Report Agent Answer Quality - RAGAS (scientific results)

Each evaluated local LLM answered the frozen `gemma4` query-plan questions from identical contextual-RAG evidence (Ollama-context hybrid, K=5). Answers are compared to `gpt-oss:20b` reference answers using RAGAS answer-only metrics.

- Plan source (frozen): `gemma4`
- Reference generator / ground truth: `gpt-oss:20b`
- RAGAS scorer: `gpt-oss:20b` + `nomic-embed-text` embeddings
- Unique questions: 6

## Metrics

| Metric | Compared against | Meaning |
| --- | --- | --- |
| faithfulness | answer vs contexts | Grounding in retrieved evidence |
| answer_relevancy | answer vs question | Addresses the planned question |
| answer_correctness | answer vs gpt-oss | Correctness vs reference answer |
| semantic_similarity | answer vs gpt-oss | Meaning closeness to reference |

## Main results - answer metrics by model (mean)

| model | faithfulness | answer_relevancy | answer_correctness | semantic_similarity |
| --- | --- | --- | --- | --- |
| gemma4 | 0.841 | 0.560 | 0.554 | 0.929 |
| llama3 | 0.459 | 0.704 | 0.548 | 0.937 |
| mistral | 0.624 | 0.720 | 0.515 | 0.934 |
| phi4 | 0.549 | 0.681 | 0.603 | 0.944 |
| qwen2.5 | 0.366 | 0.723 | 0.539 | 0.927 |

![Answer metrics by model](charts/ragas_answer_metrics_by_model.svg)

## Leaderboard (mean of answer metrics)

| model | mean_answer_score |
| --- | --- |
| gemma4 | 0.721 |
| mistral | 0.698 |
| phi4 | 0.694 |
| llama3 | 0.662 |
| qwen2.5 | 0.639 |


In [15]:

import matplotlib.pyplot as plt

csv_path = path("outputs_dir") / "ragas_per_question.csv"
if not csv_path.exists():
    print("No ragas_per_question.csv yet.")
else:
    scored = pd.read_csv(csv_path)

    if scored.empty:
        print("No data to visualize.")
    else:
        # Select only columns related to metrics and model
        metric_cols = [col for col in ANSWER_METRICS if col in scored.columns]
        models = scored["model"].unique()

        # Instead of .show(), save figures to disk and print the file path
        # Plot metric distributions for each model
        fig, axs = plt.subplots(len(metric_cols), 1, figsize=(8, 4 * len(metric_cols)))
        if len(metric_cols) == 1:
            axs = [axs]
        for i, metric in enumerate(metric_cols):
            for model in models:
                data = scored[scored["model"] == model][metric].dropna()
                axs[i].hist(data, alpha=0.5, bins=20, label=model)
            axs[i].set_title(f"Distribution of {metric}")
            axs[i].set_xlabel(metric)
            axs[i].set_ylabel("Count")
            axs[i].legend()
        plt.tight_layout()
        dist_path = path("outputs_dir") / "ragas_metric_distributions.png"
        plt.savefig(dist_path)
        print(f"Saved metric distribution plot to: {dist_path}")

        # Bar chart: mean metric by model
        means = scored.groupby("model")[metric_cols].mean()
        ax = means.plot(kind="bar", figsize=(10, 6))
        plt.title("Average RAGAS Metrics by Model")
        plt.ylabel("Mean value")
        plt.xticks(rotation=45)
        plt.legend(title="Metric")
        plt.tight_layout()
        bar_path = path("outputs_dir") / "ragas_metric_means.png"
        plt.savefig(bar_path)
        print(f"Saved bar chart of mean metrics to: {bar_path}")



Saved metric distribution plot to: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\ragas_metric_distributions.png
Saved bar chart of mean metrics to: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\ragas_metric_means.png


## Optional — single-row smoke (sanity check)

Quick inspection of one already-scored / cached triple.

In [16]:
CASE_ID, QID, CAND = "hms_borderline_01", "Q1", "mistral"
frozen = load_frozen_contexts(CASE_ID, QID)
cand = load_answers(CAND, CASE_ID).get("answers", {}).get(QID, {})
ref = load_answers(REFERENCE_MODEL, CASE_ID).get("answers", {}).get(QID, {})
print("frozen contexts:", None if frozen is None else len(frozen.get("hits", [])))
print("candidate chars:", len((cand.get("answer") or "")))
print("reference chars:", len((ref.get("answer") or "")))
if frozen and cand.get("answer") and ref.get("answer"):
    print("\nQ:", frozen["question"][:200], "…")

frozen contexts: 5
candidate chars: 1768
reference chars: 941

Q: Given the prediction probability (0.5001) is extremely close to the decision threshold (0.5), what are the clinical guidelines for interpreting 'near-threshold' results in student mental health screen …


In [26]:
from IPython.display import Image, display
data = [{"gemma4",0.721},
{"mistral",0.698},
{"phi4",0.694},
{"llama3",0.662},
{"qwen2.5",0.639}]
import matplotlib.pyplot as plt
import pandas as pd

# Correct the data construction: list of dicts should have correct key-value pairs
data = [
    {"model": "gemma4", "mean_answer_score": 0.721},
    {"model": "mistral", "mean_answer_score": 0.698},
    {"model": "phi4", "mean_answer_score": 0.694},
    {"model": "llama3", "mean_answer_score": 0.662},
    {"model": "qwen2.5", "mean_answer_score": 0.639},
]

df = pd.DataFrame(data)

# Draw barplot
plt.figure(figsize=(8, 5))
plt.bar(df["model"], df["mean_answer_score"], color="skyblue")
plt.xlabel("Model")
plt.ylabel("Mean Answer Score")
plt.title("Leaderboard (mean of answer metrics)")
plt.ylim(0, 1)
plt.tight_layout()
barplot_path = path("outputs_dir") / "single_row_leaderboard.png"
plt.savefig(barplot_path)
print(f"Saved leaderboard barplot to: {barplot_path}")

Saved leaderboard barplot to: C:\Users\MossabBatal\Desktop\psychology_mas\report_agent_eval\outputs\single_row_leaderboard.png
